# Train Schematica's symbol detector

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. When it finishes (about an hour), your browser downloads `model.onnx` and `metrics.json`.
   Put both in `ml/weights/` in the repo.

The repo must be public for the clone below. If it is private, upload a zip of it instead and unzip it to `/content/schematica-circuits`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Shaj2x/schematica-circuits.git
%cd /content/schematica-circuits/ml
# Colab already has PyTorch; this adds Ultralytics, ONNX export and the pipeline itself.
!pip install -q -e ".[ocr]" "ultralytics>=8.3,<9" onnx onnxslim

## Get CGHD

**Option A (default): Kaggle**, through `kagglehub`. Public datasets need no login. If the name below is wrong, search Kaggle for "CGHD" and paste the `owner/name` from the dataset's URL.

**Option B: your own copy.** Upload the CGHD archive to Google Drive, set `CGHD_ZIP` to its path, and the next cell uses that instead.

In [ ]:
KAGGLE_DATASET = "johannesbayer/cghd1152"
CGHD_ZIP = ""  # e.g. "/content/drive/MyDrive/cghd.zip" for option B

import pathlib, subprocess

if CGHD_ZIP:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["unzip", "-q", "-o", CGHD_ZIP, "-d", "data/cghd"], check=True)
    CGHD = pathlib.Path("data/cghd")
else:
    import kagglehub

    CGHD = pathlib.Path(kagglehub.dataset_download(KAGGLE_DATASET))
print(CGHD, sum(1 for _ in CGHD.rglob("*.xml")), "annotation files")

## Prepare, train, evaluate, export

`prepare_dataset.py` prints how every CGHD label was mapped. Read that list: a class that should be ours but landed in `other` means CGHD renamed it, so add it to `CLASS_MAP`.

In [ ]:
!python training/prepare_dataset.py --cghd "{CGHD}" --out data/yolo

In [ ]:
# yolov8s at 1024 px: ~1 h on a T4. For a quicker first try use --model yolov8n.pt --epochs 50.
!python training/train.py --data data/yolo/data.yaml --model yolov8s.pt --epochs 100 --imgsz 1024

In [ ]:
import json

metrics = json.load(open("weights/metrics.json"))
print("ONNX model on held-out drafters:")
print({k: v for k, v in metrics["test_onnx"].items() if k != "per_class"})
for name, m in metrics["test_onnx"]["per_class"].items():
    print(f"  {name:15s} mAP50 {m['mAP50']:.3f}  recall {m['recall']:.3f}")

In [ ]:
# Whole-pipeline check on rendered circuits (wire tracing + OCR + layout, with this model).
!python training/evaluate.py --synthetic 100

In [ ]:
from google.colab import files

files.download("weights/model.onnx")
files.download("weights/metrics.json")